# Construction PPE Detector
A simple end-to-end notebook: check the GPU, download the public dataset, train YOLO, evaluate the test split, and export the model. In Colab choose **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable GPU from Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q ultralytics==8.3.221
from ultralytics import YOLO
from pathlib import Path
from importlib.metadata import files as package_files
import json, shutil

## Train
Ultralytics downloads the Construction-PPE dataset automatically. It contains 1,416 images split into train, validation, and test sets, with 11 PPE-related classes. YOLO11n is used because it trains quickly and is suitable for a junior portfolio project.

In [ ]:
run_dir = Path('/content/runs/ppe-yolo11n')
if run_dir.exists():
    shutil.rmtree(run_dir)

model = YOLO('yolo11n.pt')
model.train(
    data='construction-ppe.yaml',
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    project='/content/runs',
    name='ppe-yolo11n',
    exist_ok=True,
    seed=42,
)

## Evaluate on the test split
These are the numbers to discuss in an interview. They come from images that were not used for training or early stopping.

In [ ]:
best_path = run_dir / 'weights/best.pt'
assert best_path.exists(), f'Training checkpoint not found: {best_path}'
best_model = YOLO(best_path)
eval_dir = Path('/content/evaluation/test')
if eval_dir.exists():
    shutil.rmtree(eval_dir)
metrics = best_model.val(
    data='construction-ppe.yaml', split='test', imgsz=640, device=0, plots=True,
    project=str(eval_dir.parent), name=eval_dir.name, exist_ok=True,
)
summary = {
    'precision': round(float(metrics.box.mp), 4),
    'recall': round(float(metrics.box.mr), 4),
    'mAP50': round(float(metrics.box.map50), 4),
    'mAP50_95': round(float(metrics.box.map), 4),
}
print(json.dumps(summary, indent=2))
metrics_path = eval_dir / 'metrics.json'
metrics_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

## Preview predictions

In [ ]:
dataset_root = Path('/content/datasets/construction-ppe')
test_images = sorted((dataset_root / 'images/test').glob('*'))
assert test_images, 'Could not find test images'
prediction_dir = Path('/content/predictions/examples')
if prediction_dir.exists():
    shutil.rmtree(prediction_dir)
best_model.predict(
    test_images[:6], conf=0.35, save=True,
    project=str(prediction_dir.parent), name=prediction_dir.name, exist_ok=True,
)

## Export and download results

In [ ]:
onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True)
release = Path('/content/construction-ppe-results')
if release.exists():
    shutil.rmtree(release)
release.mkdir(parents=True)
shutil.copy2(best_path, release / 'best.pt')
shutil.copy2(onnx_path, release / 'best.onnx')
shutil.copy2(metrics_path, release / 'metrics.json')
license_file = next(
    path for path in package_files('ultralytics')
    if str(path).replace('\\', '/').endswith('licenses/LICENSE')
)
shutil.copy2(license_file.locate(), release / 'LICENSE')
for name in ('confusion_matrix.png', 'PR_curve.png', 'F1_curve.png'):
    source = eval_dir / name
    if source.exists(): shutil.copy2(source, release / name)
zip_path = Path('/content/construction-ppe-results.zip')
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive('/content/construction-ppe-results', 'zip', release)
print('Created /content/construction-ppe-results.zip')

In [ ]:
from google.colab import files
files.download('/content/construction-ppe-results.zip')